# CogVideoX inference server: image + text to video (free Kaggle GPU, single card, 4-bit)

Takes a prompt (text) together with one image (the first frame) to generate a video
([CogVideoX-5b-I2V](https://huggingface.co/THUDM/CogVideoX-5b-I2V)): the prompt controls how things
move, the image controls how they look. **First frame only: Agnes-style first+last frame control
is not supported** (there is no `last_frame`).

**It always uses a single GPU with 4-bit (nf4) quantization.** I first tried splitting the text
encoder and the diffusion transformer across two GPUs to save memory, but **measured three different
low-level failures in a row** (device mismatch, CUDA illegal memory access). Cross-card sharding
appears to be unstable in bitsandbytes quantization itself (8-bit/4-bit quantization is mainly
designed and tested for a single card), not a small bug in this code, so I dropped multi-GPU sharding
in favour of the stable single-card route with a tighter memory margin. Choosing `GPU T4 x2` is fine;
the second card simply stays idle.

The local `cogvideox_client.py` needs no changes for this server; only `COGVIDEOX_BASE_URL` in your
`.env` has to be set to the URL this notebook prints. Calls must include `first_frame` (a local image
path is enough; the client converts it to a data URI automatically).

**Two things to do on Kaggle the first time, otherwise later steps fail confusingly:**
1. Your account must be phone-verified (Kaggle settings, `Phone Verification`) to use GPUs.
2. In this notebook's right-hand Settings panel, turn the `Internet` toggle on (it is off by default).

**Usage**
1. Upload to [Kaggle Notebooks](https://www.kaggle.com/code) (`File` -> `Import Notebook`).
2. Right-hand `Settings` -> `Accelerator`: choose `GPU T4 x2` or `GPU P100` (only the first card is used);
   turn `Internet` on.
3. Right-hand `Add-ons` -> `Secrets`: add `SERVER_API_KEY` and `NGROK_AUTH_TOKEN`
   (names must match what step 3 reads).
4. Run every cell from top to bottom.
5. The last cell prints `COGVIDEOX_BASE_URL=...`; put it in your `.env`.
6. Keep the notebook running; your local code can only reach it while it is running.

**Whenever you change this file** (for example to apply a later fix), remember to replace the
corresponding cell's code in your already-running Kaggle session, or re-upload the whole file.
Editing the file locally does not sync to Kaggle; the running session must be updated by hand.

## 1. Install packages

In [ ]:
!pip install -q diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg pillow
!pip install -q fastapi uvicorn nest-asyncio pyngrok
!pip install -q bitsandbytes

# The preinstalled torchao version often does not match the API the latest diffusers expects
# (e.g. ImportError: cannot import name 'FqnToConfig' from 'torchao.quantization').
# We only use fp16 / bitsandbytes inference and do not need torchao, so removing it is the cleanest fix.
!pip uninstall -y torchao -q

## 2. Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Open Settings -> Accelerator on the right, choose GPU T4 x2 or GPU P100, "
    "save, and re-run."
)
props = torch.cuda.get_device_properties(0)
print(f"GPU 0: {props.name}, {props.total_memory / 1024**3:.1f} GB")
if torch.cuda.device_count() > 1:
    print(f"({torch.cuda.device_count() - 1} additional GPU(s) detected. This notebook does not use them; "
          "leaving them idle is fine. Multi-GPU sharding proved unstable in testing, so it was "
          "reverted to a single card.)")

## 3. Configuration: API key and ngrok token (reads Kaggle Secrets)

In the right-hand `Add-ons` -> `Secrets` panel add `SERVER_API_KEY` and `NGROK_AUTH_TOKEN`
(names must match what the next cell reads), and make sure this notebook is allowed to access them.

In [ ]:
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()
SERVER_API_KEY = _secrets.get_secret("SERVER_API_KEY")
NGROK_AUTH_TOKEN = _secrets.get_secret("NGROK_AUTH_TOKEN")

MODEL_NAME = "THUDM/CogVideoX-5b-I2V"

## 4. Load the model (the first run downloads the weights; the 5B model is much larger than 2B, so it takes longer)

Always GPU 0 only and always 4-bit quantization (nf4). See the intro: sharding across two cards was
measured to be unstable with this bitsandbytes + accelerate combination (two different low-level
errors in a row: device mismatch and CUDA illegal memory access), so this uses a single card for stability.

In [ ]:
import gc
import os

# After many consecutive generations GPU memory fragments: there is free space in theory, but the
# allocator cannot find one contiguous block for a new tensor and raises CUDA out of memory
# (even when it looks like there is room). This setting lets PyTorch's allocator reuse freed
# blocks more flexibly. It must be set before CUDA is initialized.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import T5EncoderModel, BitsAndBytesConfig as TransformersBnbConfig
from diffusers import (
    CogVideoXImageToVideoPipeline,
    CogVideoXTransformer3DModel,
    BitsAndBytesConfig as DiffusersBnbConfig,
)

# GPU 0 only, 4-bit (nf4) only: dual-GPU sharding was unstable in testing (see the intro).
quant_kwargs = dict(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)

text_encoder = T5EncoderModel.from_pretrained(
    MODEL_NAME, subfolder="text_encoder", torch_dtype=torch.float16,
    quantization_config=TransformersBnbConfig(**quant_kwargs),
    device_map={"": 0},
)
gc.collect()
torch.cuda.empty_cache()

transformer = CogVideoXTransformer3DModel.from_pretrained(
    MODEL_NAME, subfolder="transformer", torch_dtype=torch.float16,
    quantization_config=DiffusersBnbConfig(**quant_kwargs),
    device_map={"": 0},
)
gc.collect()
torch.cuda.empty_cache()

pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    MODEL_NAME, text_encoder=text_encoder, transformer=transformer,
    torch_dtype=torch.float16,
)
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
pipe.vae.to("cuda:0")   # The VAE is not quantized and small (a few hundred MB): keep it on the same card as everything else
gc.collect()
torch.cuda.empty_cache()
print("Model loaded (image + text to video, CogVideoX-5b-I2V, single card, 4-bit).")

## 5. Define the API server (create job / poll status / download file)

The response shape deliberately mirrors the Agnes video API (`video_id`, `status`, `metadata.url`),
so the local `cogvideox_client.py` needs no parsing changes.

In [ ]:
import base64
import io
import threading
import time
import traceback
import uuid
from pathlib import Path
from queue import Queue, Empty
from typing import Optional

import requests as http_requests
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel
from PIL import Image
from diffusers.utils import export_to_video

# Kaggle's writable working directory is /kaggle/working (not Colab's /content).
OUTPUT_DIR = Path("/kaggle/working/cogvideox_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PUBLIC_BASE_URL = ""  # Filled in by the next-but-one cell once the ngrok tunnel exists

# Resolution used by the official CogVideoX-5b-I2V examples; the input image is resized to it.
IMAGE_WIDTH, IMAGE_HEIGHT = 720, 480
NUM_INFERENCE_STEPS = 50

tasks = {}         # video_id -> {"status", "progress", "error", "file"}
job_queue = Queue()

app = FastAPI()


class CreateVideoRequest(BaseModel):
    model: Optional[str] = None
    prompt: str
    mode: str = "image"
    seconds: str = "5"
    aspect_ratio: str = "16:9"
    seed: Optional[int] = None
    # Accepts either a data URI (data:image/...;base64,...) or a public URL.
    first_frame: Optional[str] = None


def check_auth(authorization: Optional[str]):
    if authorization != f"Bearer {SERVER_API_KEY}":
        raise HTTPException(status_code=401, detail="invalid api key")


def seconds_to_frames(seconds_str: str) -> int:
    """CogVideoX runs at 8 fps and the frame count must be 4k+1; clamped to 9..49 (about 1-6 seconds)."""
    try:
        seconds = float(seconds_str)
    except (TypeError, ValueError):
        seconds = 5.0
    raw = round(seconds * 8)
    k = max(2, min(12, round((raw - 1) / 4)))
    return 4 * k + 1


def load_first_frame(value: str) -> Image.Image:
    """Load first_frame (data URI or public URL) as a PIL image resized to the model's resolution."""
    if value.startswith("data:"):
        _, b64data = value.split(",", 1)
        raw = base64.b64decode(b64data)
    else:
        resp = http_requests.get(value, timeout=60)
        resp.raise_for_status()
        raw = resp.content
    image = Image.open(io.BytesIO(raw)).convert("RGB")
    return image.resize((IMAGE_WIDTH, IMAGE_HEIGHT))


def _clear_cache_on_last_step(pipe_obj, step_index, timestep, callback_kwargs):
    """Measured: right after the 50 denoising steps finish, the VAE decode (latents -> frames)
    hits CUDA OOM (denoising leaves only ~250 MB free because 4-bit quantization still nearly
    fills the card). On the last denoising step, before decode starts, proactively clear the cache
    to squeeze out as much room as possible for decode."""
    if step_index == NUM_INFERENCE_STEPS - 1:
        gc.collect()
        torch.cuda.empty_cache()
    return callback_kwargs


def worker_loop():
    while True:
        try:
            video_id, req = job_queue.get(timeout=1)
        except Empty:
            continue
        tasks[video_id]["status"] = "processing"
        result = None
        frames = None
        try:
            if not req.first_frame:
                raise ValueError("This server requires first_frame on every request (image + text to video)")
            num_frames = seconds_to_frames(req.seconds)
            generator = None
            if req.seed is not None:
                generator = torch.Generator(device="cuda:0").manual_seed(int(req.seed))
            image = load_first_frame(req.first_frame)
            result = pipe(
                prompt=req.prompt, image=image,
                height=IMAGE_HEIGHT, width=IMAGE_WIDTH,
                num_frames=num_frames, guidance_scale=6,
                num_inference_steps=NUM_INFERENCE_STEPS, generator=generator,
                callback_on_step_end=_clear_cache_on_last_step,
            )
            frames = result.frames[0]
            out_path = OUTPUT_DIR / f"{video_id}.mp4"
            export_to_video(frames, str(out_path), fps=8)
            tasks[video_id].update(status="completed", progress=100, file=str(out_path))
        except Exception as e:
            # Failures happen on a background thread and do not make the cell itself fail, so
            # storing the error only in tasks[] would leave Kaggle's own output showing nothing.
            # Print the full traceback so the real error is visible directly on the Kaggle side.
            print(f"[error] {video_id} generation failed:")
            traceback.print_exc()
            tasks[video_id].update(status="failed", error=str(e)[:500])
        finally:
            # After several consecutive jobs GPU memory fragments. After each job, explicitly drop
            # the intermediate results and force a collection so the memory really goes back to
            # the allocator, reducing the chance that the next job hits OOM.
            del result, frames
            gc.collect()
            torch.cuda.empty_cache()


threading.Thread(target=worker_loop, daemon=True).start()


@app.post("/v1/videos")
def create_video(req: CreateVideoRequest, authorization: Optional[str] = Header(None)):
    check_auth(authorization)
    video_id = uuid.uuid4().hex
    tasks[video_id] = {"status": "queued", "progress": 0}
    job_queue.put((video_id, req))
    return {"video_id": video_id, "id": video_id, "task_id": video_id}


@app.get("/v1/videos/{video_id}")
def get_video_status(video_id: str, authorization: Optional[str] = Header(None)):
    check_auth(authorization)
    task = tasks.get(video_id)
    if not task:
        raise HTTPException(status_code=404, detail="video_id not found")
    out = {"status": task["status"], "progress": task.get("progress", 0)}
    if task["status"] == "completed":
        out["metadata"] = {"url": f"{PUBLIC_BASE_URL}/v1/videos/{video_id}/file"}
    if task["status"] == "failed":
        out["error"] = task.get("error")
    return out


@app.get("/v1/videos/{video_id}/file")
def get_video_file(video_id: str, authorization: Optional[str] = Header(None)):
    check_auth(authorization)
    task = tasks.get(video_id)
    if not task or task.get("status") != "completed":
        raise HTTPException(status_code=404, detail="video not ready")
    return FileResponse(task["file"], media_type="video/mp4")


print("API defined (image + text to video). The next cell starts the ngrok tunnel and the server.")

## 6. Open the ngrok tunnel and start the server

This cell prints the public URL; put it in your `.env` as `COGVIDEOX_BASE_URL=<that URL>`.
The cell keeps running while the server is up; press the stop button on its left to shut it down.

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

try:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
except Exception as e:
    # If ngrok detects the same token already in its config it prints "already set" instead of
    # "saved", which pyngrok's string matching treats as a failure. The token itself is fine.
    if "already set" not in str(e):
        raise
    print("(ngrok authtoken already configured, skipping.)")

tunnel = ngrok.connect(8000)
PUBLIC_BASE_URL = tunnel.public_url

print("=" * 70)
print(f"COGVIDEOX_BASE_URL={PUBLIC_BASE_URL}")
print(f"COGVIDEOX_API_KEY={SERVER_API_KEY}")
print("Copy the two lines above into the .env file of the project that uses cogvideox_client.py.")
print("=" * 70)

# Do not use uvicorn.run(app, ...): it calls asyncio.run() internally, but this cell is already
# running inside an event loop, which raises
# RuntimeError: asyncio.run() cannot be called from a running event loop.
# Awaiting server.serve() attaches to the existing loop without going through asyncio.run().
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()